# Coleta Instagram Graph API -- Guia para as tasks do airflow


## Fluxos suportados
| | `auth_method = 'facebook'` | `auth_method = 'instagram'` |
|---|---|---|
| **Base URL** | `https://graph.facebook.com` | `https://graph.instagram.com` |
| **Endpoint perfil** | `/{version}/{ig_user_id}` | `/{version}/me` |
| **Endpoint mídia** | `/{version}/{ig_user_id}/media` | `/{version}/{ig_user_id}/media` |
| **Token** | `LONG_LIVED_TOKEN_FB` | `LONG_LIVED_TOKEN_IG` |

# 1. Setup

In [1]:
import requests
import json
import os
import time
from datetime import datetime
from collections import Counter
from dotenv import load_dotenv
import sys

sys.path.insert(0, os.path.abspath(".."))

In [2]:
load_dotenv()
GRAPH_VERSION = "v25.0"
AUTH_METHOD = "facebook"  # instagram ou facebook

if AUTH_METHOD == "facebook":
    BASE_URL     = "https://graph.facebook.com"
    access_token = os.getenv("LONG_LIVED_TOKEN_FB")
    ig_user_id   = os.getenv("PROFILE_ID_FB")
else:
    BASE_URL     = "https://graph.instagram.com"
    access_token = os.getenv("LONG_LIVED_TOKEN_IG")
    ig_user_id   = os.getenv("PROFILE_ID_IG")


# 2. Perfil

Única diferença entre os fluxos: o endpoint.
- **Facebook:** `/{ig_user_id}` — o `/me` retornaria o usuário do Facebook, não o Instagram
- **Instagram:** `/me` — aponta diretamente para a conta autenticada


In [3]:
def fetch_profile(base_url, graph_version, ig_user_id, access_token, auth_method):
    """
    Coleta dados do perfil Instagram.
    - facebook: usa /{ig_user_id} com fields sem account_type
    - instagram: usa /me com fields incluindo account_type
    """
    if auth_method == "facebook":
        endpoint = f"{base_url}/{graph_version}/{ig_user_id}"
        fields   = "id,username,name,biography,website,followers_count,follows_count,media_count,profile_picture_url"
    else:
        endpoint = f"{base_url}/{graph_version}/me"
        fields   = "id,username,name,biography,website,followers_count,follows_count,media_count,profile_picture_url,account_type"

    response = requests.get(endpoint, params={"fields": fields, "access_token": access_token})

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Erro {response.status_code}: {response.text}")
        return None

In [4]:
profile = fetch_profile(BASE_URL, GRAPH_VERSION, ig_user_id, access_token, AUTH_METHOD)
print(json.dumps(profile, indent=2, ensure_ascii=False))

{
  "id": "17841477255589822",
  "username": "technews24.7",
  "name": "Notícias Tech 24/7",
  "biography": "noticias tech",
  "followers_count": 2,
  "follows_count": 25,
  "media_count": 2,
  "profile_picture_url": "https://scontent-for2-2.xx.fbcdn.net/v/t51.82787-15/549498754_17844264411575187_2607132828623270266_n.jpg?_nc_cat=103&ccb=1-7&_nc_sid=7d201b&_nc_ohc=-lpy4Y5E9qAQ7kNvwFbBWLA&_nc_oc=Adnhdz_AtAllh6ir_vrwZXvSYz5-XhhuX4Bym48V-eu8Hk29fJilDeQIfRdbslkPVTxNetYNFt8TdSbD54lLKEr5&_nc_zt=23&_nc_ht=scontent-for2-2.xx&edm=AL-3X8kEAAAA&_nc_gid=_GBwVJ60P7UYhyvMXkrSaQ&oh=00_Afy8vziKlsy87yesB4yaXvro2e1u8YoK0jbWv7T4ZUE5dg&oe=69BF3D00"
}



# 3. Posts / Mídia — `fetch_all_posts()`

Coleta todos os posts do perfil com paginação completa por cursor.
O endpoint `/{ig_user_id}/media` funciona igual nos dois fluxos — só muda o `base_url`.

- `limit=100` é o máximo seguro por página
- A paginação avança pelo campo `paging.next` retornado na resposta
- O Graph API retorna no máximo **10.000 posts** por perfil

In [5]:
MEDIA_FIELDS = "id,caption,media_type,media_url,thumbnail_url,permalink,timestamp,like_count,comments_count"

def fetch_all_posts(base_url, graph_version, ig_user_id, access_token, limit=100):
    """
    Coleta todos os posts do perfil com paginação por cursor.
    Funciona igual para os fluxos facebook e instagram.
    Retorna lista com todos os posts coletados.
    """
    all_posts = []
    page_num  = 1
    next_url  = (
        f"{base_url}/{graph_version}/{ig_user_id}/media"
        f"?fields={MEDIA_FIELDS}&limit={limit}&access_token={access_token}"
    )

    while next_url:
        response = requests.get(next_url)

        if response.status_code != 200:
            print(f"Erro na página {page_num}: {response.status_code}")
            print(response.text)
            break

        data   = response.json()
        posts  = data.get("data", [])
        paging = data.get("paging", {})

        all_posts.extend(posts)
        print(f"Página {page_num:>3} | +{len(posts):>3} posts | Total: {len(all_posts):>5}")

        next_url = paging.get("next")
        page_num += 1

    print(f"\nTotal de posts coletados: {len(all_posts)}")
    return all_posts

In [6]:
all_posts = fetch_all_posts(BASE_URL, GRAPH_VERSION, ig_user_id, access_token)

# Distribuição por tipo de mídia
tipos = Counter(p.get("media_type") for p in all_posts)
print("\nDistribuição por tipo de mídia:")
for tipo, qtd in tipos.most_common():
    print(f"  {tipo:<20}: {qtd}")

Página   1 | +  2 posts | Total:     2

Total de posts coletados: 2

Distribuição por tipo de mídia:
  IMAGE               : 2


# 4. Comentários + Replies — `fetch_comments()` e `fetch_replies()`

Coleta comentários de um post e as replies de cada comentário.
O endpoint `/{media_id}/comments` e `/{comment_id}/replies` funcionam igual nos dois fluxos.

- Comentários são paginados — a função já trata isso

>Insight de post postado **antes** da conversão da conta para Business/Creator retorna erro — esse comportamento já foi observado nos testes.

In [7]:
COMMENT_FIELDS = "id,text,timestamp,like_count,username"
REPLY_FIELDS   = "id,text,timestamp,username"

def fetch_comments(base_url, graph_version, media_id, access_token, limit=100):
    """
    Coleta todos os comentários de um post com paginação.
    Funciona igual para os fluxos facebook e instagram.
    Retorna lista de comentários.
    """
    all_comments = []
    next_url = (
        f"{base_url}/{graph_version}/{media_id}/comments"
        f"?fields={COMMENT_FIELDS}&limit={limit}&access_token={access_token}"
    )

    while next_url:
        response = requests.get(next_url)

        if response.status_code != 200:
            print(f"Erro ao coletar comentários de {media_id}: {response.status_code}")
            print(response.text)
            break

        data         = response.json()
        comments     = data.get("data", [])
        all_comments.extend(comments)
        next_url     = data.get("paging", {}).get("next")

    return all_comments


def fetch_replies(base_url, graph_version, comment_id, access_token, limit=100):
    """
    Coleta todas as replies de um comentário.
    Funciona igual para os fluxos facebook e instagram.
    Retorna lista de replies.
    """
    all_replies = []
    next_url = (
        f"{base_url}/{graph_version}/{comment_id}/replies"
        f"?fields={REPLY_FIELDS}&limit={limit}&access_token={access_token}"
    )

    while next_url:
        response = requests.get(next_url)

        if response.status_code != 200:
            print(f"Erro ao coletar replies de {comment_id}: {response.status_code}")
            break

        data        = response.json()
        replies     = data.get("data", [])
        all_replies.extend(replies)
        next_url    = data.get("paging", {}).get("next")

    return all_replies

In [8]:
# Testa com o primeiro post coletado
sample_post_id = all_posts[0]["id"] if all_posts else "SEU_MEDIA_ID_AQUI"
print(f"Coletando comentários do post: {sample_post_id}")

comments = fetch_comments(BASE_URL, GRAPH_VERSION, sample_post_id, access_token)
print(f"Comentários encontrados: {len(comments)}")

if comments:
    print("\nPrimeiro comentário:")
    print(json.dumps(comments[0], indent=2, ensure_ascii=False))

    # Replies do primeiro comentário
    replies = fetch_replies(BASE_URL, GRAPH_VERSION, comments[0]["id"], access_token)
    print(f"\nReplies do primeiro comentário: {len(replies)}")
    if replies:
        print(json.dumps(replies[0], indent=2, ensure_ascii=False))

Coletando comentários do post: 18406296661186436
Comentários encontrados: 3

Primeiro comentário:
{
  "id": "18412438912127005",
  "text": "Top",
  "timestamp": "2026-02-27T13:20:38+0000",
  "like_count": 0,
  "username": "cortesfilosofiabr"
}

Replies do primeiro comentário: 0


# 5. Insights de Posts — `fetch_post_insights()`

Coleta métricas de desempenho por post. O endpoint `/{media_id}/insights` funciona igual nos dois fluxos.

Métricas por `media_type` (v25.0 — `impressions` removido desde v22):
- **IMAGE / CAROUSEL_ALBUM:** `reach, saved, shares, total_interactions`
- **VIDEO:** `reach, saved, shares, total_interactions, plays`
- **STORY:** `reach, shares, replies, total_interactions`

> Posts publicados **antes** da conversão para conta Business/Creator retornam erro — esse comportamento é esperado e já foi observado nos testes. A função registra o erro e continua.

In [9]:
# Métricas disponíveis por tipo de mídia (v25.0)
METRICS_MAP = {
    "IMAGE":          "reach,saved,shares,total_interactions,views,profile_activity",
    "CAROUSEL_ALBUM": "reach,saved,shares,total_interactions,views,profile_activity",
    "VIDEO":          "reach,saved,shares,total_interactions,views,ig_reels_avg_watch_time,ig_reels_video_view_total_time",
    "STORY":          "reach,shares,replies,total_interactions,navigation,views,profile_activity",
}

def fetch_post_insights(base_url, graph_version, post, access_token):
    """
    Coleta insights de um post individual.
    Funciona igual para os fluxos facebook e instagram.
    Retorna dict com as métricas mescladas ao post, ou None em caso de erro.
    """
    media_id   = post["id"]
    media_type = post.get("media_type", "IMAGE")
    metrics    = METRICS_MAP.get(media_type, "reach,saved,shares,total_interactions")

    url      = f"{base_url}/{graph_version}/{media_id}/insights?metric={metrics}&access_token={access_token}"
    response = requests.get(url)

    if response.status_code == 200:
        insight_data = {
            i["name"]: (i["values"][0]["value"] if i.get("values") else i.get("value"))
            for i in response.json().get("data", [])
        }
        return {**post, **insight_data}
    else:
        err = response.json().get("error", {}).get("message", response.text)
        return {"error": err, "id": media_id}


def fetch_all_post_insights(base_url, graph_version, posts, access_token):
    """
    Itera sobre uma lista de posts e coleta os insights de cada um.
    Retorna dois lists: posts enriquecidos e posts com erro.
    """
    enriched = []
    errors   = []

    for post in posts:
        result = fetch_post_insights(base_url, graph_version, post, access_token)

        if "error" in result:
            errors.append(result)
            print(f"ERROR: {result['id']} | {post.get('media_type','?'):<15} | {result['error']}")
        else:
            enriched.append(result)
            print(f"{result['id']} | {post.get('media_type','?'):<15} | reach: {result.get('reach', 'N/A')}")

    print(f"\nCom insights: {len(enriched)} | Erros: {len(errors)}")
    return enriched, errors

In [10]:
# Testa com os 10 primeiros posts
posts_com_insights, posts_com_erro = fetch_all_post_insights(
    BASE_URL, GRAPH_VERSION, all_posts[:10], access_token
)

if posts_com_insights:
    print("\nExemplo de post enriquecido:")
    print(json.dumps(posts_com_insights[0], indent=2, ensure_ascii=False))

18406296661186436 | IMAGE           | reach: 2
ERROR: 18091336891776984 | IMAGE           | Invalid parameter

Com insights: 1 | Erros: 1

Exemplo de post enriquecido:
{
  "id": "18406296661186436",
  "caption": "#### ATENÇÃO ####\nO CAFEZINHO PRETO NO COPO AMERICANO É OFICIALMENTE A BEBIDA DOS DEVS",
  "media_type": "IMAGE",
  "media_url": "https://scontent-for2-2.cdninstagram.com/v/t51.82787-15/583465111_17853982278575187_3238836366871602539_n.jpg?stp=dst-jpg_e35_tt6&_nc_cat=108&ccb=7-5&_nc_sid=18de74&efg=eyJlZmdfdGFnIjoiRkVFRC5iZXN0X2ltYWdlX3VybGdlbi5DMyJ9&_nc_ohc=5mgCa9PxebcQ7kNvwG5UfwI&_nc_oc=AdmweJQL2n_YEVIyxXSdA0fUPjjQMHJKXJu--WttBHW81nEtOag8T8DBkEL4sWahWOB69m94vNrLgAxfiDrzU6Si&_nc_zt=23&_nc_ht=scontent-for2-2.cdninstagram.com&edm=AM6HXa8EAAAA&_nc_gid=X6Rac8xIcdKMbzCYFlfxBw&oh=00_AfyVCXhsthumTCVQq94B6u6Q0cMXBqnLrJ-edqUZqLo8Nw&oe=69BF39A3",
  "permalink": "https://www.instagram.com/p/DRKwmmPkfgS/",
  "timestamp": "2025-11-17T18:45:30+0000",
  "like_count": 4,
  "comments_count": 

# 6. Insights da Conta — `fetch_account_insights()` e `fetch_audience_insights()`

Duas funções separadas porque usam parâmetros distintos da API:

**`fetch_account_insights`** — métricas de interação por período (`reach`, `profile_views`, etc.)
- Usa `period=day` + `since/until` (Unix timestamps)
- Funciona igual nos dois fluxos com `metric_type=total_value`
- Fluxo Instagram: não suporta `follows_and_unfollows` — use apenas `reach,profile_views,total_interactions`

**`fetch_audience_insights`** — métricas demográficas (`follower_demographics`, `engaged_audience_demographics`)
- Usa `period=lifetime` + `timeframe` + `breakdown` (uma chamada por breakdown)
- `timeframe` aceito na v25: `this_month` ou `this_week`
- Retorna `data: []` se a conta tiver menos de 100 seguidores/engajamentos no período


In [11]:
def fetch_account_insights(base_url, graph_version, ig_user_id, access_token,
                           since: datetime, until: datetime, auth_method="facebook"):
    """
    Coleta métricas de interação da conta por período.
    - facebook: suporta reach, profile_views, total_interactions, follows_and_unfollows
    - instagram: suporta reach, profile_views, total_interactions
                 (follows_and_unfollows não disponível neste fluxo)
    Retorna lista de métricas com valores por dia.
    """
    if auth_method == "facebook":
        metrics = "reach,profile_views,total_interactions,follows_and_unfollows"
    else:
        metrics = "reach,profile_views,total_interactions"

    since_ts = int(time.mktime(since.timetuple()))
    until_ts = int(time.mktime(until.timetuple()))

    url = (
        f"{base_url}/{graph_version}/{ig_user_id}/insights"
        f"?metric={metrics}"
        f"&period=day"
        f"&metric_type=total_value"
        f"&since={since_ts}"
        f"&until={until_ts}"
        f"&access_token={access_token}"
    )

    response = requests.get(url)

    if response.status_code == 200:
        return response.json().get("data", [])
    else:
        print(f"Erro {response.status_code}: {response.text}")
        return []


def fetch_audience_insights(base_url, graph_version, ig_user_id, access_token,
                            breakdowns=("country", "city", "age", "gender"),
                            timeframe="this_month"):
    """
    Coleta métricas demográficas da audiência.
    Faz uma requisição por breakdown (exigência da API).
    Funciona igual para os dois fluxos.
    Retorna dict {breakdown: data}.
    """
    results = {}

    for breakdown in breakdowns:
        url = (
            f"{base_url}/{graph_version}/{ig_user_id}/insights"
            f"?metric=engaged_audience_demographics,follower_demographics"
            f"&period=lifetime"
            f"&metric_type=total_value"
            f"&timeframe={timeframe}"
            f"&breakdown={breakdown}"
            f"&access_token={access_token}"
        )

        response = requests.get(url)

        if response.status_code == 200:
            results[breakdown] = response.json().get("data", [])
            print(f"breakdown={breakdown} | registros: {len(results[breakdown])}")
        else:
            print(f"ERROR breakdown={breakdown}: {response.status_code} {response.text}")
            results[breakdown] = []

    return results

In [12]:
# Insights de interação — últimos 30 dias
since = datetime(2026, 2, 16)
until = datetime.now()

print("Coletando insights de interação da conta...")
account_insights = fetch_account_insights(
    BASE_URL, GRAPH_VERSION, ig_user_id, access_token,
    since=since, until=until, auth_method=AUTH_METHOD
)

for metric in account_insights:
    print(f"\n  Métrica : {metric['name']} (period={metric.get('period')})")
    for v in metric.get("values", [])[:5]:
        print(f"    {v['end_time'][:10]} : {v['value']}")

Coletando insights de interação da conta...

  Métrica : reach (period=day)

  Métrica : profile_views (period=day)

  Métrica : total_interactions (period=day)


In [13]:
# Insights de audiência — demográficos
# Retorna data:[] se conta tiver < 100 seguidores/engajamentos no período
print("Coletando insights de audiência...")
audience_insights = fetch_audience_insights(
    BASE_URL, GRAPH_VERSION, ig_user_id, access_token,
    breakdowns=("country", "city", "age", "gender"),
    timeframe="this_month"
)

print("\nResultado por breakdown:")
print(json.dumps(audience_insights, indent=2, ensure_ascii=False))

Coletando insights de audiência...
breakdown=country | registros: 0
breakdown=city | registros: 0
breakdown=age | registros: 0
breakdown=gender | registros: 0

Resultado por breakdown:
{
  "country": [],
  "city": [],
  "age": [],
  "gender": []
}
